In [1]:
# reuse of code from previous tutorials to load data and model
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512,512),
            nn.ReLU(),
            nn.Linear(512,10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

In [2]:
# hyperparameters --> control of model optimization process

epochs = 5              # number of iterations through the whole dataset
batch_size = 64         # number of examples before network parameters are updated
learning_rate = 1e-3    # how much to update model parameters each batch --> carefull

In [5]:
# optimization loop
    # loss function
loss_fn = nn.CrossEntropyLoss()     # initialize loss-function

    # optimizer --> adjusting model parameters
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    # steps in the trainings loop
    # 1. reset gradients to zero --> optimizer.zero_grad()
    # 2. backpropagate the prediction loss --> loss.backwards()
    # 3. update parameters --> optimizer.step()

In [7]:
# full implementation
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()           # set model to training mode
    for batch, (X, y) in enumerate(dataloader):
        # compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    model.eval()            # set model to evaluation mode
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # evaluation model with toch.no_grad --> no gradient computation
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


# outer epoche loop
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 20
for t in range(epochs):
    print(f"Epoch {t+1}\n------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
------------------
loss: 2.316667 [   64/60000]
loss: 2.296946 [ 6464/60000]
loss: 2.272973 [12864/60000]
loss: 2.256813 [19264/60000]
loss: 2.250245 [25664/60000]
loss: 2.216015 [32064/60000]
loss: 2.226239 [38464/60000]
loss: 2.197507 [44864/60000]
loss: 2.192205 [51264/60000]
loss: 2.159015 [57664/60000]
Test Error: 
 Accuracy: 43.9%, Avg loss: 2.151259 

Epoch 2
------------------
loss: 2.170398 [   64/60000]
loss: 2.151632 [ 6464/60000]
loss: 2.095080 [12864/60000]
loss: 2.107069 [19264/60000]
loss: 2.056937 [25664/60000]
loss: 1.999598 [32064/60000]
loss: 2.017953 [38464/60000]
loss: 1.946297 [44864/60000]
loss: 1.947022 [51264/60000]
loss: 1.875558 [57664/60000]
Test Error: 
 Accuracy: 59.5%, Avg loss: 1.869233 

Epoch 3
------------------
loss: 1.906779 [   64/60000]
loss: 1.867314 [ 6464/60000]
loss: 1.753664 [12864/60000]
loss: 1.796556 [19264/60000]
loss: 1.683642 [25664/60000]
loss: 1.639755 [32064/60000]
loss: 1.647495 [38464/60000]
loss: 1.561775 [44864/60000]
los